# Novel Attack Experiment 1: DECP — Detector-Evasive Classifier Poisoning

## Attack
**Threat model**: attacker injects poisoned samples into the live data stream before the classifier
and detector see them (the `poison_fn` slot in `StreamingPipeline`).

**Simplest DECP variant** (zero knowledge — no model weights, no gradients):

For each novel-class sample in the batch:
1. **Evasion** — linearly interpolate toward the nearest known-class centroid in input space
   by factor α.  This moves the latent embedding closer to a known-class prototype, pulling
   the NCM distance below the detector's drift threshold.
2. **Corruption** — replace the label with the *farthest* known-class centroid index.
   If the classifier trains online, it learns a wrong association; if it is static the
   attack still suppresses the drift signal that would otherwise trigger adaptation.

Required attacker knowledge: class centroids in input space (computable from the public data distribution).

## Comparison via `StreamingPipeline`
| Detector | Latent space | Hypothesis |
|---|---|---|
| `ContrastiveNCMAdapter` | Contrastive AE — same-class samples cluster tightly | More structured → input-space interpolation may not move latent embedding below threshold |
| `PlainNCMAdapter` | Plain AE (MSE only) — less structure | Encoder is more linear → input-space projection transfers to latent space |

## Data
Synthetic 4-class blob dataset (sklearn). Classes {0,1,2} = known. Class 3 = novel (drift).
Stream: 20 clean batches → 30 novel batches.

In [ ]:
import io
import os
import sys
import copy
import contextlib

sys.path.insert(0, os.path.join(os.path.abspath(''), '../..'))

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn as nn
from sklearn.datasets import make_blobs
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

from src.detectors.contrastive_ncm import (
    ContrastiveNCMDetector,
    DriftAutoencoder,
    NCMClassifier,
    train_plain_autoencoder,
)
from src.streaming_pipeline import ContrastiveNCMAdapter, StreamingPipeline

# ── Hyperparameters ───────────────────────────────────────────────────────────
SEED          = 42
INPUT_DIM     = 20
HIDDEN_DIM    = 16
LATENT_DIM    = 8
N_KNOWN       = 3       # known classes in reference data
BATCH_SIZE    = 128
N_CLEAN       = 20      # clean known-class batches in stream
N_NOVEL       = 30      # novel-class batches in stream
ALPHA         = 0.85    # DECP interpolation weight toward centroid
FIT_EPOCHS    = 60
FIT_LR        = 5e-4
CAL_Q         = 0.95    # quantile for drift threshold calibration

np.random.seed(SEED)
torch.manual_seed(SEED)
print('Setup complete.')

## 1. Data

In [ ]:
X_all, y_all = make_blobs(
    n_samples=8000,
    n_features=INPUT_DIM,
    centers=N_KNOWN + 1,   # classes 0,1,2 known; class 3 novel
    cluster_std=1.5,
    random_state=SEED,
)
X_all = StandardScaler().fit_transform(X_all).astype(np.float32)
y_all = y_all.astype(np.int64)

known_mask = y_all < N_KNOWN
X_known, y_known = X_all[known_mask], y_all[known_mask]
X_novel, y_novel = X_all[~known_mask], y_all[~known_mask]

X_ref, X_test_k, y_ref, y_test_k = train_test_split(
    X_known, y_known, test_size=0.3, random_state=SEED, stratify=y_known
)

# Input-space centroids: the only attacker knowledge required
class_centroids = np.vstack([X_ref[y_ref == c].mean(axis=0) for c in range(N_KNOWN)])

print(f'Reference   : {len(X_ref):,}')
print(f'Known test  : {len(X_test_k):,}')
print(f'Novel       : {len(X_novel):,}')
print(f'Centroids   : {class_centroids.shape}')

## 2. Detector & Classifier Definitions

In [ ]:
class MLPClassifier:
    """Two-hidden-layer MLP. fit(X,y) / predict(X) interface for StreamingPipeline."""

    def __init__(self, input_dim: int, n_classes: int, hidden: int = 64,
                 lr: float = 1e-3, epochs: int = 40):
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden),   nn.ReLU(),
            nn.Linear(hidden, n_classes),
        )
        self._lr     = lr
        self._epochs = epochs

    def fit(self, X: np.ndarray, y: np.ndarray) -> None:
        opt = torch.optim.Adam(self.net.parameters(), lr=self._lr)
        ds  = DataLoader(
            TensorDataset(torch.tensor(X), torch.tensor(y, dtype=torch.long)),
            batch_size=128, shuffle=True,
        )
        self.net.train()
        for _ in range(self._epochs):
            for xb, yb in ds:
                opt.zero_grad()
                nn.CrossEntropyLoss()(self.net(xb), yb).backward()
                opt.step()
        self.net.eval()

    def predict(self, X: np.ndarray) -> np.ndarray:
        with torch.no_grad():
            return self.net(torch.tensor(X)).argmax(dim=1).numpy()


class PlainNCMAdapter:
    """
    Drop-in DriftDetector using a plain (MSE-only) autoencoder + NCM.
    Mirrors ContrastiveNCMAdapter's interface so both can run through StreamingPipeline.
    """

    def __init__(self, input_dim: int, hidden_dim: int, latent_dim: int,
                 batch_size: int = 128, fit_epochs: int = 60, fit_lr: float = 5e-4):
        self._ae            = DriftAutoencoder(input_dim, hidden_dim, latent_dim)
        self._ncm           = NCMClassifier(lambda_1=0.1)
        self.drift_threshold = 1.0
        self._bs            = batch_size
        self._epochs        = fit_epochs
        self._lr            = fit_lr

    def _loader(self, X: np.ndarray, y: np.ndarray) -> DataLoader:
        return DataLoader(
            TensorDataset(torch.tensor(X), torch.tensor(y, dtype=torch.long)),
            batch_size=self._bs, shuffle=True,
        )

    def fit(self, X: np.ndarray, y: np.ndarray) -> None:
        loader = self._loader(X, y)
        with contextlib.redirect_stdout(io.StringIO()):
            train_plain_autoencoder(self._ae, loader, epochs=self._epochs, lr=self._lr)
        self._ae.eval()
        all_h, all_y = [], []
        with torch.no_grad():
            for xb, yb in loader:
                all_h.append(self._ae.encode(xb))
                all_y.append(yb)
        num_classes = int(np.max(y)) + 1
        self._ncm.fit(torch.cat(all_h), torch.cat(all_y), num_classes)

    def detect(self, X: np.ndarray, y: np.ndarray) -> bool:
        self._ae.eval()
        with torch.no_grad():
            h = self._ae.encode(torch.tensor(X))
        dists = self._ncm._compute_distance(h).min(dim=1).values
        return bool((dists > self.drift_threshold).any())

    def adapt(self, X: np.ndarray, y: np.ndarray) -> None:
        loader = self._loader(X, y)
        with contextlib.redirect_stdout(io.StringIO()):
            train_plain_autoencoder(
                self._ae, loader,
                epochs=max(1, self._epochs // 10),
                lr=self._lr,
            )
        self._ae.eval()


print('Classes defined.')

## 3. Train Reference Models

In [ ]:
# ── MLP classifier ────────────────────────────────────────────────────────────
print('Training MLP classifier...')
classifier = MLPClassifier(INPUT_DIM, N_KNOWN)
classifier.fit(X_ref, y_ref)
ref_acc = np.mean(classifier.predict(X_test_k) == y_test_k)
print(f'  Accuracy on known test set: {ref_acc:.3f}')

# ── ContrastiveNCM detector ───────────────────────────────────────────────────
print('\nTraining ContrastiveNCM detector...')
ncm_det = ContrastiveNCMDetector(
    input_dim=INPUT_DIM, hidden_dim=HIDDEN_DIM, latent_dim=LATENT_DIM,
    drift_threshold=1.0, concept_threshold=3.5,
)
ncm_adapter_ref = ContrastiveNCMAdapter(
    ncm_det, batch_size=BATCH_SIZE, fit_epochs=FIT_EPOCHS, fit_lr=FIT_LR
)
ncm_adapter_ref.fit(X_ref, y_ref)

# Calibrate drift threshold at CAL_Q-th percentile of reference distances
with torch.no_grad():
    h_cal = ncm_det.encode(torch.tensor(X_ref))
    cal_d  = ncm_det.ncm._compute_distance(h_cal).min(dim=1).values
ncm_det.drift_threshold = float(cal_d.quantile(CAL_Q))
print(f'  Calibrated drift threshold: {ncm_det.drift_threshold:.4f}')

# ── PlainNCM detector ─────────────────────────────────────────────────────────
print('\nTraining PlainNCM detector...')
plain_adapter_ref = PlainNCMAdapter(
    INPUT_DIM, HIDDEN_DIM, LATENT_DIM, fit_epochs=FIT_EPOCHS, fit_lr=FIT_LR
)
plain_adapter_ref.fit(X_ref, y_ref)

plain_adapter_ref._ae.eval()
with torch.no_grad():
    h_plain_cal = plain_adapter_ref._ae.encode(torch.tensor(X_ref))
    plain_cal_d = plain_adapter_ref._ncm._compute_distance(h_plain_cal).min(dim=1).values
plain_adapter_ref.drift_threshold = float(plain_cal_d.quantile(CAL_Q))
print(f'  Calibrated drift threshold: {plain_adapter_ref.drift_threshold:.4f}')

## 4. DECP Attack Definition & Sanity Check

In [ ]:
class DECPAttack:
    """
    Detector-Evasive Classifier Poisoning — simplest (zero-knowledge) form.

    Evasion   : x' = (1-α)·x + α·centroid_{nearest}   [input-space interpolation]
    Corruption: y' = argmax_c dist(x, centroid_c)      [farthest centroid label]

    Attacker needs only: known-class centroids in input space.
    No deployed model weights or gradient access required.
    """

    def __init__(self, class_centroids: np.ndarray, alpha: float = 0.85):
        self.centroids = class_centroids  # (n_known, input_dim)
        self.alpha     = alpha

    def __call__(self, X: np.ndarray, y: np.ndarray):
        dists_c  = np.linalg.norm(X[:, None, :] - self.centroids[None, :, :], axis=2)  # (N, C)
        nearest  = dists_c.argmin(axis=1)
        farthest = dists_c.argmax(axis=1)

        X_proj  = (1 - self.alpha) * X + self.alpha * self.centroids[nearest]
        y_flip  = farthest.astype(np.int64)

        return X_proj.astype(np.float32), y_flip


attack = DECPAttack(class_centroids, alpha=ALPHA)

# ── Sanity check: per-sample NCM distances before and after projection ────────
sample = X_novel[:BATCH_SIZE]
X_proj, _ = attack(sample, np.zeros(BATCH_SIZE, dtype=np.int64))

def ncm_min_dists(adapter, X_arr):
    """Return per-sample minimum NCM distances using an adapter's internal components."""
    if isinstance(adapter, ContrastiveNCMAdapter):
        ae, ncm, thr = adapter._detector.autoencoder, adapter._detector.ncm, adapter._detector.drift_threshold
        ae.eval()
        with torch.no_grad():
            h = ae.encode(torch.tensor(X_arr).to(adapter._detector.device))
    else:
        ae, ncm, thr = adapter._ae, adapter._ncm, adapter.drift_threshold
        ae.eval()
        with torch.no_grad():
            h = ae.encode(torch.tensor(X_arr))
    return ncm._compute_distance(h).min(dim=1).values.cpu().numpy(), thr

for name, adapter in [('ContrastiveNCM', ncm_adapter_ref), ('PlainNCM', plain_adapter_ref)]:
    d_before, thr = ncm_min_dists(adapter, sample)
    d_after,  _   = ncm_min_dists(adapter, X_proj)
    evaded = (d_after <= thr).mean()
    print(f'{name:16s} | threshold={thr:.4f}'
          f'  dist_before={d_before.mean():.4f}  dist_after={d_after.mean():.4f}'
          f'  sample evasion={evaded:.1%}')

## 5. Streaming Experiment via `StreamingPipeline`

Four conditions: {ContrastiveNCM, PlainNCM} × {clean stream, DECP-attacked stream}.

Each condition uses a `deepcopy` of the trained adapter so runs are independent.
The MLP classifier is shared (static — trained once, never updated in the stream).

**Primary metric**: how many of the 30 novel batches are flagged as drift.

In [ ]:
def build_stream(X_clean, y_clean, X_novel, y_novel, n_clean, n_novel, batch_size, seed=SEED):
    """Return a list of (X_batch, y_batch): n_clean clean batches then n_novel novel batches."""
    rng = np.random.RandomState(seed)
    batches = []
    for _ in range(n_clean):
        idx = rng.choice(len(X_clean), batch_size, replace=True)
        batches.append((X_clean[idx], y_clean[idx]))
    for _ in range(n_novel):
        idx = rng.choice(len(X_novel), batch_size, replace=True)
        batches.append((X_novel[idx], y_novel[idx]))
    return batches


def run_pipeline(adapter, batches, poison_fn=None):
    """Wrap batches in StreamingPipeline, run, return per-batch drift flags."""
    pipeline = StreamingPipeline(
        data_stream=iter(batches),
        classifier=classifier,
        detector=adapter,
        poison_fn=poison_fn,
    )
    metrics = pipeline.run()
    return [r.drift_detected for r in metrics.batch_results]


results = {}
for det_name, base_adapter in [('ContrastiveNCM', ncm_adapter_ref),
                                 ('PlainNCM',       plain_adapter_ref)]:
    for attacked in [False, True]:
        adapter = copy.deepcopy(base_adapter)
        pfn     = attack if attacked else None
        label   = f"{det_name} {'+ DECP' if attacked else '(clean)'}"

        batches = build_stream(X_test_k, y_test_k, X_novel, y_novel, N_CLEAN, N_NOVEL, BATCH_SIZE)
        drift_per_batch = run_pipeline(adapter, batches, pfn)
        results[(det_name, attacked)] = drift_per_batch

        false_alarms   = sum(drift_per_batch[:N_CLEAN])
        novel_detected = sum(drift_per_batch[N_CLEAN:])
        print(f'{label:45s} | false alarms: {false_alarms}/{N_CLEAN}  '
              f'novel detected: {novel_detected}/{N_NOVEL}')

## 6. Results

In [ ]:
rows = []
for (det, attacked), drifts in results.items():
    novel_det = sum(drifts[N_CLEAN:])
    rows.append({
        'Detector'              : det,
        'Attack'                : f'DECP (α={ALPHA})' if attacked else 'None',
        'False alarms (clean)'  : f"{sum(drifts[:N_CLEAN])}/{N_CLEAN}",
        'Novel batches detected': f"{novel_det}/{N_NOVEL}",
        'Evasion rate'          : f"{1 - novel_det/N_NOVEL:.0%}" if attacked else '—',
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))
print(f'\nAlpha={ALPHA}  |  Batch size={BATCH_SIZE}  |  Clean batches={N_CLEAN}  |  Novel batches={N_NOVEL}')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 7), sharey=True)
config = [
    ('ContrastiveNCM', False, axes[0, 0]),
    ('ContrastiveNCM', True,  axes[0, 1]),
    ('PlainNCM',       False, axes[1, 0]),
    ('PlainNCM',       True,  axes[1, 1]),
]

for det, attacked, ax in config:
    drifts   = results[(det, attacked)]
    xs       = list(range(len(drifts)))
    ax.bar(xs[:N_CLEAN], drifts[:N_CLEAN], color='#4C72B0', alpha=0.85, label='Known-class batch')
    ax.bar(xs[N_CLEAN:], drifts[N_CLEAN:], color='#DD8452', alpha=0.90, label='Novel-class batch')
    ax.axvline(N_CLEAN - 0.5, color='black', linestyle='--', linewidth=1.2)
    ax.set_title(f"{det} {'+ DECP attack' if attacked else '(no attack)'}", fontweight='bold')
    ax.set_xlabel('Batch index')
    ax.set_ylabel('Drift detected')
    ax.set_ylim(0, 1.35)
    ax.set_yticks([0, 1])
    ax.set_yticklabels(['No', 'Yes'])
    ax.text(N_CLEAN / 2 - 0.5,      1.22, 'Clean phase', ha='center', fontsize=9, color='#4C72B0')
    ax.text(N_CLEAN + N_NOVEL / 2,  1.22, 'Novel phase',  ha='center', fontsize=9, color='#DD8452')
    ax.legend(loc='upper left', fontsize=8)

fig.suptitle(
    f'DECP Attack (α={ALPHA}) — Batch-level Drift Detection\n'
    f'Synthetic 4-class blobs · {N_CLEAN} clean + {N_NOVEL} novel batches · batch size {BATCH_SIZE}',
    fontsize=12, fontweight='bold',
)
plt.tight_layout()
plt.savefig('decp_experiment_1_results.png', dpi=150, bbox_inches='tight')
plt.show()